# Agent | Tools

**1. 도구 (Tool)**

https://docs.langchain.com/oss/python/integrations/tools/index#tools-and-toolkits

> "LLM이 사용할 수 있는 구체적인 기술이나 장비"

LLM은 기본적으로 학습된 데이터 내에서만 답변할 수 있으며, 실시간 정보나 정확한 수학 계산에는 취약하다. **Tool**은 이러한 한계를 보완하기 위해 LLM에게 쥐여주는 외부 기능이다.

* **역활:** 외부 API 호출, 웹 검색, 코드 실행, 파일 시스템 접근 등 LLM이 직접 할 수 없는 작업을 대신 수행한다.
* **예시:**
* `Google Search`: 최신 정보를 검색한다.
* `Calculator`: 정확한 수학 계산을 수행한다.
* `Python REPL`: 파이썬 코드를 작성하고 실행한다.



**2. 에이전트 (Agent)**

> "도구를 언제, 어떻게 사용할지 결정하는 두뇌"

**Agent**는 LLM을 추론 엔진(Reasoning Engine)으로 사용하여 사용자의 요청을 해결하기 위한 계획을 세우고 실행하는 주체이다. 단순히 정해진 코드를 순서대로 실행하는 것이 아니라, 상황에 따라 유연하게 행동을 결정한다.

* **역활:** 사용자의 질문을 분석하고, 어떤 **Tool**이 필요한지 판단(Thought)하고, 해당 도구를 실행(Action)한 뒤, 그 결과(Observation)를 보고 다음 행동을 결정하거나 최종 답변을 내놓는다.
* **작동 방식 (ReAct 패턴 예시):**
1. **질문:** "현재 서울 날씨에 맞는 옷차림 추천해줘."
2. **생각(Thought):** "서울의 현재 날씨를 먼저 알아야 한다." -> `Search` 도구 선택
3. **행동(Action):** `Search("서울 현재 날씨")` 실행
4. **관찰(Observation):** "서울 기온 5도, 맑음"이라는 결과 획득
5. **생각(Thought):** "5도면 코트나 패딩이 필요하다."
6. **최종 답변:** "현재 서울은 5도이므로 코트나 가벼운 패딩을 추천합니다."

**Agent와 Tools의 상호작용**

1. **Agent가 입력을 받음**: 사용자의 요청을 LLM으로 분석.
2. **적합한 Tool 선택**: 요청을 처리하는 데 가장 적합한 Tool을 선택.
3. **Tool 실행 및 결과 반환**: Tool을 실행하고 결과를 받아 사용자에게 응답.

In [1]:
%pip install -Uqqq langchain_openai langchain_community langchain_tavily langgraph wikipedia numexpr arxiv ddgs

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

## Tools

In [5]:
import importlib, pkgutil # 모듈 동적 로드 / 패키지 탐색 유틸

# langchain_community.tools 패키지 로드
package = importlib.import_module('langchain_community.tools')

# 해당 패키지 경로 아래의 하위 모듈들을 하나씩 순회
for module in pkgutil.iter_modules(package.__path__):
    print(module.name) # 각 모듈(도구) 이름 출력

ainetwork
amadeus
arxiv
asknews
audio
azure_ai_services
azure_cognitive_services
bearly
bing_search
brave_search
cassandra_database
clickup
cogniswitch
connery
convert_to_openai
databricks
dataforseo_api_search
dataherald
ddg_search
e2b_data_analysis
edenai
eleven_labs
few_shot
file_management
financial_datasets
github
gitlab
gmail
golden_query
google_books
google_cloud
google_finance
google_jobs
google_lens
google_scholar
google_serper
google_trends
graphql
human
ifttt
interaction
jina_search
jira
json
memorize
merriam_webster
metaphor_search
mojeek_search
multion
nasa
nuclia
office365
openai_dalle_image_generation
openapi
openweathermap
passio_nutrition_ai
playwright
plugin
polygon
powerbi
pubmed
render
requests
riza
scenexplain
searchapi
searx_search
semanticscholar
shell
slack
sleep
spark_sql
sql_database
stackexchange
steam
steamship_image_generation
tavily_search
vectorstore
wikidata
wikipedia
wolfram_alpha
yahoo_finance_news
you
youtube
zapier
zenguard


### Wikipedia Tool


In [ ]:
from langchain_community.tools import WikipediaQueryRun          # 위키피디아 질문 실행 Tool
from langchain_community.utilities import WikipediaAPIWrapper    # 위키피디아 검색/요약 API 요청 래퍼 클래스

# 위키피디아 API 래퍼를 Tool에 연결
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
print(wiki_tool.run('Physical AI')) # 위키피디아 검색/요약 결과 출력

Page: Physical artificial intelligence
Summary: Physical artificial intelligence or physical AI refers to artificial intelligence (AI) systems that perceive, reason about and act within the physical world. These systems generally combine AI models with sensors, control systems, actuators and physical machines such as robots or autonomous vehicles. Physical AI overlaps with embodied artificial intelligence, robotics and autonomous systems, but it emphasizes the complete process of perceiving an environment, motion planning an action and physically executing the task to perform work. This differs from digital AI or generative AI (GenAI), which primarily stays in the information or digital realm.
The term became increasingly prominent during the AI boom in the 2020s as AI development expanded from primarily digital applications toward humanoid robots, self-driving vehicles, smart factories and other autonomous machines. Its boundaries are not standardized, and it is often treated as a con

In [12]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from pprint import pprint

messages = [('human', '걸그룹 튜이드 멤버 알려줘')]

llm = init_chat_model('gpt-5.4-mini')
# print(llm.invoke('걸그룹 튜이드 멤버 알려줘')) # 최신정보 알지 못함

agent = create_agent(
    model = llm,
    tools = [wiki_tool]
)

response = agent.invoke({'messages': messages})

pprint(response)

{'messages': [HumanMessage(content='걸그룹 튜이드 멤버 알려줘', additional_kwargs={}, response_metadata={}, id='613486cd-b83f-416a-a1ea-1148cdaf7101'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 173, 'total_tokens': 192, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHKoEMIlwqRarhbdLUgXtJ2oKkLWh', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04133-42df-7830-9e5e-460052576901-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'T-ara members'}, 'id': 'call_UlOvfkDNa81Px2LPOMdULEXB', '

In [13]:
print(response['messages'][-1].content)

걸그룹 **티아라(T-ara)** 멤버는 현재 기준으로 **4명**입니다.

- **큐리(Qri)**
- **은정(Eunjung)**
- **효민(Hyomin)**
- **지연(Jiyeon)**

원하시면 **과거 멤버까지 포함한 전체 멤버 변동**도 정리해드릴게요.


### load_tools

**load_tools 사용가능 목록**

라이브러리를 통해 제공되는 외부 tool들을 langchain-community에서 통합하여 사용할 수 있다.
웬만한 기능들은 langchain 생태계 내에서 쉽게 쓸 수 있다.

https://docs.langchain.com/oss/python/integrations/providers/overview

https://docs.langchain.com/oss/python/integrations/tools

| 도구 이름        | 기능 예시               |
|------------------|------------------------|
| llm-math         | LLM 기반 수학 계산     |
| wikipedia        | 위키백과 검색          |
| serpapi          | 구글 검색 API          |
| requests_get     | HTTP GET 요청          |
| requests_post    | HTTP POST 요청         |
| arxiv            | arXiv 논문 검색        |
| pubmed           | PubMed 논문 검색       |
| dalle            | DALL-E 이미지 생성     |
| bing_search      | Bing 검색              |
| duckduckgo_search| DuckDuckGo 검색        |

### arxiv

In [4]:
import requests  # HTTP 요청 보내는 라이브러리
import xml.etree.ElementTree as ET   # XML 응답 파싱
from langchain_core.tools import tool # Langchain Tool 생성 데코레이터

@tool
def search_arxiv(arxiv_id: str) -> str:
    """ arXib 논문 ID로 제목, 저자, 초록을 조회합니다."""

    url = "https://export.arxiv.org/api/query"
    response = response.get(
        url,
        params = {
            "id_list": arxiv_id, # 논문 ID
            "max_results": 1     # 결과 1개
        },
        timeout = 10             # 응답 대기시간
    )
  
    response.raise_for_status()  # 요청 실패시 예외 발생

    root = ET.fromstring(response.text) # XML 문자열을 받아 Element 객체로 반환

    ns = {"atom": "http://www.w3.org/2005/Atom"}  #  arXiv 응답의 XML 네임 스페이스
    entry = root.find("atom:entry", ns)           # 논문 정보가 담긴 entry 태그

    if entry is None:
        return "논문 정보를 찾을 수 없습니다."

    title = entry.findtext("atom:title", namespaces = ns).strip()     # 논문 제목 추출
    summary = entry.findtext("atom:summary", namespaces = ns).strip()  # 요약 정보 추출
    authors = [  # 저자 추출
        author.findtext("atom:name", namespaces=ns)
        for author in entry.findall("atom:author", ns)
    ]

    return f"""
    제목: {title}
    저자: {', '.join(authors)}
    초록: {summary}
    """


In [6]:
from pprint import pprint

In [9]:
from langchain_community.agent_toolkits.load_tools import load_tools

llm = init_chat_model('gpt-4.1-mini')


tools = load_tools(['wikipedia', 'llm-math'], llm = llm)

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = """
당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변해 주세요.
단, 숫자게산은 반드시 llm-math 도구를 사용해서 답변에 활용해야 합니다."""
)

response = agent.invoke({
    'messages': [
        ('human', '유클리드 기하에서 평행선 공준이 지켜지지 않는 기하는?')
    ]
})

pprint(response)
print("=" * 100)
pprint(response['messages'][-1])


{'messages': [HumanMessage(content='유클리드 기하에서 평행선 공준이 지켜지지 않는 기하는?', additional_kwargs={}, response_metadata={}, id='2622c97a-a2b7-41d5-a16a-6f0ca03933e0'),
              AIMessage(content='유클리드 기하에서 평행선 공준이 지켜지지 않는 기하는 비유클리드 기하입니다. 비유클리드 기하에는 크게 두 가지가 있는데,\n\n1. 쌍곡기하학: 평행선 공준이 지켜지지 않고, 한 점에서 여러 개의 평행선이 그어질 수 있습니다.\n2. 타원기하학: 평행선이 아예 존재하지 않습니다.\n\n이 두 기하학은 각각 평행선 공준이 성립하지 않는 대표적인 비유클리드 기하학입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 185, 'total_tokens': 314, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_e67debc4e8', 'id': 'chatcmpl-EHNiUHWJF3aIHmuAcokWoNl4Hk6uU', 'servi

### duckduckgo
https://reference.langchain.com/python/langchain-community/tools/ddg_search/tool/DuckDuckGoSearchRun

DuckDuckGo는 개인정보 추적 없이 웹 검색을 제공하는 검색 엔진으로,
LangChain에서는 이를 외부 최신 정보 검색용 Tool로 활용한다.

실시간 웹 검색 가능
→ LLM의 knowledge cutoff 이후 이슈(뉴스, 화제, 트렌드)에 대응 가능
로그인/API 키 불필요
→ 실습·교육 환경에서 바로 사용 가능
프라이버시 중심
→ 사용자 검색 이력 추적 없음

LangChain에서 제공하는 DuckDuckGo Tool 차이
- DuckDuckGoSearchRun
    - 검색 결과를 하나의 텍스트 요약으로 반환
    - 빠른 질의응답용에 적합
- DuckDuckGoSearchResults
    - 검색 결과를 리스트(제목, 링크, 스니펫 등 구조화) 형태로 반환
    - 에이전트가 여러 결과를 비교·판단해야 할 때 유리

In [11]:
# 덕덕고 검색 Tool 2종류
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults

ddgs = DuckDuckGoSearchRun() # 검색 결과를 텍스트 요약 형태로 반환
print(ddgs.invoke("Trump's first name?")) # 문자열 출력

ddgs2 = DuckDuckGoSearchRun() # 검색 결과를 구조화된 리스트로 반환
print(ddgs.invoke("Trump's first name?")) # 제목/링크/스니펫 정보등의 리스트로 반환

2.2Licensing the Trump name. 2.3Side ventures. 2.3.1Trump University. Donald Trump 's first tenure as the president of the United States began on January 20, 2017, when he was inaugurated as the 45th president, and ended on January 20, 2021. Donald Trump ... Donald John Trump (born June 14, 1946 [1][2]) is an American politician, businessman, and television personality who has been the 47th president of the United States since 2025. Donald Trump's real name is not "Donald Drumpf." Trump is the surname his parents gave him at birth, and he has never used "Drumpf." However, historical records indicate that some of his... 6 hours ago · Donald Trump (born June 14, 1946, New York, New York, U.S.) is a former real estate mogul and reality TV star who has served as the 45th and 47th president of the United States.
2.2Licensing the Trump name. 2.3Side ventures. 2.3.1Trump University. Donald Trump 's first tenure as the president of the United States began on January 20, 2017, when he was inaug

In [14]:
from pprint import pprint

llm = init_chat_model(
    'gpt-5.6-luna',
    reasoning_effort='none'
)

# DDGS 도구 로드
tools = [ddgs2]

agent = create_agent(
    llm,
    tools,
    system_prompt='모르는 정보가 있으면 ddgs tool을 사용해 검색해.'
)

response = agent.invoke({
    'messages': [
        ('human', 'GS25 민음사 빵')
    ]
})

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='GS25 민음사 빵', additional_kwargs={}, response_metadata={}, id='6d8670d1-79ff-453a-914c-46a3a6e1a2e4'),
              AIMessage(content='GS25에서 판매하는 **민음사 빵**은 민음사와 GS25가 협업한 독서 콘셉트의 베이커리 상품으로 알려져 있습니다. 보통 민음사 대표 도서·문학 디자인을 활용한 패키지가 특징이며, 상품 구성과 판매 여부는 점포별로 다를 수 있습니다.\n\n정확한 **제품명·가격·판매 매장**을 확인하려면 GS25 앱 **우리동네GS**에서 ‘민음사’ 또는 ‘빵’을 검색하거나 가까운 매장에 문의해 보세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 179, 'total_tokens': 307, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHNtPIIhSUnn5qskYd8Qyl23Pii45', 'service_tier': 'default', 'finish_rea

tavily-search
https://docs.langchain.com/oss/python/integrations/tools/tavily_search

In [16]:
from langchain_tavily import TavilySearch # Tavily 검색 Tool

tavily_tool = TavilySearch(
    max_results = 3,
    topic = 'general',    # general/news/finance 등 선택
    include_images = True,   # 이미지 URL 함께 반환
    search_depth = 'advanced' # basic/advanced (advanced는 더 깊게 찾음)
)
tavily_tool.invoke('현재 대한민국에서 가장 핫한 이슈가 뭐야?')

{'query': '현재 대한민국에서 가장 핫한 이슈가 뭐야?',
 'follow_up_questions': None,
 'answer': None,
 'images': ['https://i.ytimg.com/vi/FfFY8UKEW5s/maxresdefault.jpg',
  'https://pimg3.daara.co.kr/kidd/photo/2023/12/28/1703746241_15.jpg',
  'https://lookaside.instagram.com/seo/google_widget/crawler/?media_id=3840701500981246710',
  'https://pimg3.daara.co.kr/kidd/photo/2023/12/28/1703746236_84.jpg',
  'https://pimg3.daara.co.kr/kidd/photo/2023/12/28/1703746220_100.jpg'],
 'results': [{'url': 'https://realhappyworld1009.tistory.com/717',
   'title': '2025년 11월 24일 대한민국에서 가장 이슈가 된 뉴스 3가지',
   'content': 'Title: 2025년 11월 24일 대한민국에서 가장 이슈가 된 뉴스 3가지\n## 2025년 11월 24일 대한민국에서 가장 이슈가 된 뉴스 3가지. * 중장기적으로는 (1) 남북 또는 다자 외교·안보 협력 강화, (2) 방산 및 안보 관련 기업 수혜 가능성, (3) 지정학 리스크 증가 시 투자 위축 등 흐름이 예상됩니다. 반도체 관세 대응 협력 가능성 — 한국·대만 협력 모색. * 한국 무역부 장관이 라디오 인터뷰에서 “미 정부의 반도체 관세 부과 가능성에 대응하기 위해 대만과 한국이 협력할 여지가 있다”고 밝혔습니다. * 한국은 최근 미국과의 무역협정에서 반도체 관세 조항이 “한국이 유리한 조건을 확보했다”고 보고되었고, 대만 역시 협상 중이므로 두 나라가 전략적 협력을 통해 **관세 압박을 완화할 가능성**이

In [18]:
llm = init_chat_model(
    'gpt-5.4-mini',
    reasoning_effort='none'
)

# DDGS 도구 로드
tools = [tavily_tool]

agent = create_agent(
    llm,
    tools,
    system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답하시오.'
)

response = agent.invoke({
    'messages': '현재 AI업계에서 가장 핫한 주제는?'})

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='현재 AI업계에서 가장 핫한 주제는?', additional_kwargs={}, response_metadata={}, id='3a6d74ea-66d0-4e22-9c54-f0d4f7914d9e'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 1310, 'total_tokens': 1427, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHO3ayezHIprtChpm532elb2Aq2B9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a041f1-b7a4-7c03-abb5-af7784cc954e-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'latest hottest topics in AI industry 2026 cu

In [19]:
llm = init_chat_model(
    'gpt-5.4-mini',
    reasoning_effort='none'
)

# DDGS 도구 로드
tools = [tavily_tool]

agent = create_agent(
    llm,
    tools,
    system_prompt='''당신은 미국주식시장 분석봇입니다.
사용자가 요청한 기업에 대한 2026년 보고서를 직관적으로 분석해주세요.

# 출력형식
다음 내용을 포함해 표형식 출력 (분석기관별 레코드로 작성)

1. 분석기관명
2. 목표주가범위 (최저 ~ 최대)
3. 전망근거 키워드
4. 신뢰도 지수(1 ~ 10)당신은 현명한 챗봇입니다''')


response = agent.invoke({
    'messages': '2026년 상승할 가능성이 가장 높은 미국주식은?'})

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='2026년 상승할 가능성이 가장 높은 미국주식은?', additional_kwargs={}, response_metadata={}, id='c2b35347-e928-4850-b8b9-3ac641185152'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 1394, 'total_tokens': 1457, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHO5HwpPdbbzWi3An3Lh1WIouPo0z', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a041f3-5a23-7ae2-8292-e704ec5bff16-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': "2026 Wall Street analyst price target 

In [20]:
from IPython.display import display, Markdown

display(Markdown(response['messages'][-1].content))

아래는 **2026년 상승 가능성이 높게 거론되는 미국주식들**을 분석기관/리서치 관점으로 정리한 표입니다.  
(참고: 실제 투자 판단은 현재 주가, 실적, 밸류에이션에 따라 달라질 수 있습니다.)

| 분석기관명 | 목표주가범위 (최저 ~ 최대) | 전망근거 키워드 | 신뢰도 지수(1~10) |
|---|---:|---|---:|
| TipRanks 컨센서스 기반 | $263 ~ $312 | AI 인프라, 데이터센터 수요, 강한 매수의견, 실적 모멘텀 | 9 |
| Reuters/Wall Street 컨센서스 | $296 ~ $330 | AWS 성장, 광고사업 확대, 자동화 투자, 마진 개선 | 8 |
| Investing.com 분석 요약 | $280 ~ $300 | 관측가능성(Observability), AI 연계 성장, 기업 고객 수요 | 8 |
| Barron’s/시장 낙관론 요약 | $455 ~ $480 | AI 반도체, 인프라 확장, 고성장 섹터, 장기 수요 | 7 |
| ChartMill 스크리너 기준 | $604 ~ $972 | 인프라 경기부양, 고성장, 재무건전성, 밸류업 가능성 | 7 |

### 한줄 결론
**2026년 상승 가능성이 가장 높게 많이 언급되는 미국주식은 NVIDIA(NVDA), Amazon(AMZN), Broadcom(AVGO), Datadog(DDOG)** 쪽입니다.  
그중 **가장 강한 시장 기대는 NVDA와 AMZN**에 집중되어 있습니다.

원하시면 다음 단계로  
**“2026년 유망 미국주식 TOP 10을 표로”** 또는 **“성장주/배당주/저평가주로 나눠서”** 정리해드릴게요.

In [21]:
# eval /exec로 문자열 코드 실행
a = 10
print(eval("5+ 3 + a"))
exec("b = 10")
print(b)

18
10


In [22]:
from langchain_core.tools import tool

@tool
def simple_calculator(query: str) -> str:
    """
    산술연산을 위한 간단한 계산기 Tool
    Args:
        query: 계산식
    return
        계산식 결과값
        
    Examples:
    - simple_calculator("5 + 3 - 2") -> "계산 결과: 6"
    -simple_calculator("4 ** 2 / 8) -> 계산 결과: 2
    """

    try:
        result = eval(query)            # 문자열을 eval로 평가(결과 반환)
        return f"계산 결과 : {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"

simple_calculator


StructuredTool(name='simple_calculator', description='산술연산을 위한 간단한 계산기 Tool\nArgs:\n    query: 계산식\nreturn\n    계산식 결과값\n\nExamples:\n- simple_calculator("5 + 3 - 2") -> "계산 결과: 6"\n-simple_calculator("4 ** 2 / 8) -> 계산 결과: 2', args_schema=<class 'langchain_core.utils.pydantic.simple_calculator'>, func=<function simple_calculator at 0x000001DE37818E00>)

In [24]:
llm = init_chat_model(
    'gpt-5.4-mini',
    reasoning_effort='none'
)

# DDGS 도구 로드
tools = [tavily_tool]

agent = create_agent(
    llm,
    tools,
    system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답하시오.'
)

response = agent.invoke({
    'messages': '7+3 *8?'})

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='7+3 *8?', additional_kwargs={}, response_metadata={}, id='3d40c956-9837-469b-a054-b885a80b3296'),
              AIMessage(content='7 + 3 × 8 = 7 + 24 = **31**', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 1303, 'total_tokens': 1323, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 1152, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHOXVhg3P1fEEORju9IxiigOz2Z3g', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0420e-0ce7-7e81-bf57-bc1284b5b3fe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1303, 'output_tokens'

In [28]:
import os
import json
import requests

from langchain_core.tools import tool


OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')


@tool
def get_current_weather(city_name='seoul', units='metric'):
    """
    OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수

    Args:
        - city_name: str 날씨 정보를 가져올 도시 이름. 반드시 영문으로 작성하세요.
            - 변환 예시:
                - 서울 -> Seoul
                - 충남, 충청남도 -> Chungcheongnam-do
                - 부산 -> Busan
        - units: str 온도 단위를 설정하는 문자열
            - metric(기본값: 섭씨, 미터)
            - imperial(화씨, 야드)

    Return:
        - str: JSON 형식으로 변환된 현재 날씨 정보
    """

    url = (
        f'https://api.openweathermap.org/data/2.5/weather'
        f'?q={city_name}&appid={OPENWEATHER_API_KEY}&units={units}'
    )

    response = requests.get(url)
    data = response.json()
    weather_info = {}

    if response.status_code == 200:
        weather_description = data['weather'][0]['description']
        temp = data['main']['temp']
        temp_feels_like = data['main']['feels_like']
        humidity = data['main']['humidity']

        weather_info = {
            'city': city_name,
            'description': weather_description,
            'temperature': temp,
            'temperature_feels_like': temp_feels_like,
            'humidity': humidity
        }

    else:
        weather_info = {
            'city': city_name,
            'description': 'Not Found',
            'temperature': 'Not Found',
            'temperature_feels_like': 'Not Found',
            'humidity': 'Not Found'
        }

    return json.dumps(weather_info, ensure_ascii=False)


get_current_weather



StructuredTool(name='get_current_weather', description='OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수\n\nArgs:\n    - city_name: str 날씨 정보를 가져올 도시 이름. 반드시 영문으로 작성하세요.\n        - 변환 예시:\n            - 서울 -> Seoul\n            - 충남, 충청남도 -> Chungcheongnam-do\n            - 부산 -> Busan\n    - units: str 온도 단위를 설정하는 문자열\n        - metric(기본값: 섭씨, 미터)\n        - imperial(화씨, 야드)\n\nReturn:\n    - str: JSON 형식으로 변환된 현재 날씨 정보', args_schema=<class 'langchain_core.utils.pydantic.get_current_weather'>, func=<function get_current_weather at 0x000001DE378725C0>)

In [31]:
llm = init_chat_model(
    'gpt-5.4-mini',
    reasoning_effort='none'
)

# 도구 로드
tools = [simple_calculator, get_current_weather]

agent = create_agent(
    llm,
    tools,
    system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답하시오.'
)

response = agent.invoke({
    'messages': [
        ('human', '강원도 사는데 오늘 옷 뭐 입을까?')
    ]
})

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content)

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='강원도 사는데 오늘 옷 뭐 입을까?', additional_kwargs={}, response_metadata={}, id='704c4a53-9b17-45ad-b70a-060d63af9de0'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 401, 'total_tokens': 426, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHOsHZKZ0xESSziqZjCLsyKNOdwau', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04221-aef9-7a70-9b08-07917e72e053-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city_name': 'Gangwon-do', 'units': 'metric'}, 'id':

In [33]:
!pip install pytz

  Using cached pytz-2026.3.post1-py2.py3-none-any.whl.metadata (22 kB)
Using cached pytz-2026.3.post1-py2.py3-none-any.whl (508 kB)



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# 한국 기준 현재 날짜/시간을 반환하는 Tool
from datetime import datetime
from pytz import timezone
from langchain_core.tools import tool


@tool
def get_current_datetime(
    format: str = '%Y-%m-%d %H:%M:%S'
) -> str:
    """
    한국 기준 현재 시각 정보를 반환하는 함수

    Args:
        format: 날짜/시각 형식 지정

    Return:
        현재 시각 문자열

    get_current_datetime() -> "2026-01-15 12:18:32"
    """

    kst = timezone('Asia/Seoul') # 한국 시간대(KST) 설정

    return datetime.now(kst).strftime(format) # 현재 서울 시간을 받아, format 형식의 문자열로 변환


get_current_datetime

StructuredTool(name='get_current_datetime', description='한국 기준 현재 시각 정보를 반환하는 함수\n\nArgs:\n    format: 날짜/시각 형식 지정\n\nReturn:\n    현재 시각 문자열\n\nget_current_datetime() -> "2026-01-15 12:18:32"', args_schema=<class 'langchain_core.utils.pydantic.get_current_datetime'>, func=<function get_current_datetime at 0x00000152C9ABC9A0>)

In [ ]:
@tool
def calculate_age(today_date: str, bitrh_date: str) -> int:
    """
    오늘날짜, 생년월일을 입력받아 나이를 계산하는 도구
    Args:
        - today_date(str): 오늘 날짜 (yyyy-mm-dd형식)
        - birth_date(str): 생년월일 (yyyy-mm-dd형식)
    Return:
        - 계산된 만나이(int)
    """

    try:
        today = datetime.strptime(today_date, '%Y-%m-%d')    # 오늘 날짜 문자열 -> datetime 변환
        birthday = datetime.strptime(bitrh_date, '%Y-%m-%d') # 생일 날짜 문자열 -> datetime 변환

        age = today.year - birthday.year # 기본 나이 계산
        # 생일이 아직 안지난 경우
        if (today.month, today.day) < (birthday.month, birthday.day):
            age -= 1 # 만나이는 -1
        return age
    except ValueError:
        return "날짜 형식이 올바르지 않습니다. yyyy-mm-dd 형식으로 전달해 주세요."

calculate_age.invoke({'today_date': '2026-08-27', 'bitrh_date': '1920-10-11'})

105

In [5]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_community.agent_toolkits.load_tools import load_tools
from pprint import pprint

In [ ]:
from langchain_community.agent_toolkits import load_tools

llm = init_chat_model(
    'gpt-5.4-mini',
    reasoning_effort='none'
)

# Wikipedia 및 사용자 정의 도구 로드
tools = load_tools(['wikipedia']) + [
    get_current_datetime,
    calculate_age
]

agent = create_agent(
    llm,
    tools,
    system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답하시오.'
)

response = agent.invoke(
    {
        'messages': [
            ('human', '트럼프 대통령은 몇 살이야?')
        ]
    },
    config={
        'recursion_limit': 10
    }
)

pprint(response)
print("=" * 100)
pprint(response['messages'][-1].content) # 30명이 api 요청해서 응답이 안옴 ㅠㅠ

TypeError: 'module' object is not callable

## Memory
agent의 checkpointer속성에 메모리객체를 대화내역을 저장한다.
- 임시저장 InMemorySaver()
- 영구저장 SqliteSaver()

### InMemorySaver

In [19]:
from langchain_tavily import TavilySearch

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver  # 대화 상태 저장

llm = init_chat_model(
    'gpt-5.4-mini',
    reasoning_effort='none'
)

tools = [TavilySearch()]

agent = create_agent(
    llm,
    tools,
    checkpointer=InMemorySaver()
)

response = agent.invoke(
    input={
        'messages': [
            ('human', '안녕! 만나서 반갑다! 나는 cap이라고 해. 넌 누구니?')
        ]
    },
    config={
        'configurable': {
            'thread_id': '100'
        }
    }
)
print()

In [23]:
response = agent.invoke(
    input = {'messages': [('human', '나는 패왕 항우다')]},
    config = {'configurable': {'thread_id': '200'}} # thread_id 200번으로 새로운 대화 시작
)
print(response['messages'][-1].content)

알겠습니다, 패왕 항우님.  
앞으로 그렇게 불러드릴게요.


### sqliteSaver

In [25]:
# Langgraph 상태 저장을 SQLite로 영속화하여 저장하는 체크포인터 패키지
%pip install -Uqqq langgraph-checkpoint-sqlite

Note: you may need to restart the kernel to use updated packages.


In [27]:
# 사용자가 재접속한 상황
from langgraph.checkpoint.sqlite import SqliteSaver # Sqlite 기반 체크포인트(Saver)
from pprint import pprint

llm = init_chat_model('gpt-5.4-mini')
tools = [TavilySearch()]

# Sqlite DB 연결을 checkpoint.db 컨텍스트로 관리
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup() # 테이블 생성 및 초기화

    # 에이전트가 상태 저장소로 checkpointer 활용
    agent = create_agent(llm,tools,checkpointer=checkpointer)

    response = agent.invoke(
        input = {'messages': [('human','오케이. 완전 이해했어! 그럼 니가 말해준 langchain, langgraph를 세줄요약해줘')]},
        config = {'configurable': {'thread_id': '100'}} # thread_id로 대화 식별
    )

    pprint(response)
    print("=" * 50)
    pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='Langchain에 대해 설명해줘.', additional_kwargs={}, response_metadata={}, id='7c27d222-b17f-4a2c-b08f-3d1dd6eac77e'),
              AIMessage(content='LangChain은 **LLM(대규모 언어 모델)을 이용한 앱을 쉽게 만들 수 있게 도와주는 프레임워크**입니다.  \n즉, 단순히 모델에 질문만 보내는 수준을 넘어서, **여러 단계의 작업, 외부 도구 연결, 문서 검색, 메모리 관리** 같은 기능을 엮어 하나의 AI 애플리케이션으로 만들 수 있게 해줍니다.\n\n## 한 줄 요약\n**LangChain = LLM을 실서비스 앱처럼 만들기 위한 연결 도구 모음**\n\n---\n\n## 왜 필요한가?\nLLM만으로는 보통 이런 한계가 있습니다.\n\n- 최신 정보에 접근하기 어려움\n- 회사 내부 문서 같은 외부 데이터 활용이 어려움\n- 여러 번의 추론을 연결하는 작업이 복잡함\n- API, DB, 검색엔진, 계산기 같은 외부 도구와 연동이 번거로움\n\nLangChain은 이런 문제를 해결하기 위해 만들어졌습니다.\n\n---\n\n## 주요 기능\n\n### 1. Prompt 관리\n프롬프트 템플릿을 체계적으로 관리할 수 있습니다.  \n예를 들어, 사용자 입력과 시스템 지시문을 조합해 일관된 형식으로 LLM에 전달할 수 있습니다.\n\n### 2. Chains\n여러 작업 단계를 연결합니다.  \n예:\n1. 사용자 질문 입력\n2. 관련 문서 검색\n3. 검색 결과 요약\n4. 최종 답변 생성\n\n이런 흐름을 하나의 파이프라인처럼 만들 수 있습니다.\n\n### 3. Agents\nLLM이 상황에 따라 어떤 도구를 쓸지 스스로 결정하게 할 수 있습니다.  \n예:\n- 계산이 필요하면 계산기 사용\n- 최신 정보가 필요하면 검색 사용\n- 파일 내용을 알아야 하면 문서 읽기 사용\n\

In [30]:
# SQLite 체크포인터(DB)의 특정 thread_id의 대화 메시지 조회
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer_tuple = checkpointer.get_tuple({"configurable": {"thread_id": "100"}})

    checkpointer_data = checkpointer_tuple.checkpoint
    messages = checkpointer_data['channel_values']['messages']

    for i, message in enumerate(messages, 1):
        msg_type = getattr(message, 'type', message.__class__.__name__)
        print(f"{i}: [{msg_type}] {message.content}")
        print()

1: [human] Langchain에 대해 설명해줘.

2: [ai] LangChain은 **LLM(대규모 언어 모델)을 이용한 앱을 쉽게 만들 수 있게 도와주는 프레임워크**입니다.  
즉, 단순히 모델에 질문만 보내는 수준을 넘어서, **여러 단계의 작업, 외부 도구 연결, 문서 검색, 메모리 관리** 같은 기능을 엮어 하나의 AI 애플리케이션으로 만들 수 있게 해줍니다.

## 한 줄 요약
**LangChain = LLM을 실서비스 앱처럼 만들기 위한 연결 도구 모음**

---

## 왜 필요한가?
LLM만으로는 보통 이런 한계가 있습니다.

- 최신 정보에 접근하기 어려움
- 회사 내부 문서 같은 외부 데이터 활용이 어려움
- 여러 번의 추론을 연결하는 작업이 복잡함
- API, DB, 검색엔진, 계산기 같은 외부 도구와 연동이 번거로움

LangChain은 이런 문제를 해결하기 위해 만들어졌습니다.

---

## 주요 기능

### 1. Prompt 관리
프롬프트 템플릿을 체계적으로 관리할 수 있습니다.  
예를 들어, 사용자 입력과 시스템 지시문을 조합해 일관된 형식으로 LLM에 전달할 수 있습니다.

### 2. Chains
여러 작업 단계를 연결합니다.  
예:
1. 사용자 질문 입력
2. 관련 문서 검색
3. 검색 결과 요약
4. 최종 답변 생성

이런 흐름을 하나의 파이프라인처럼 만들 수 있습니다.

### 3. Agents
LLM이 상황에 따라 어떤 도구를 쓸지 스스로 결정하게 할 수 있습니다.  
예:
- 계산이 필요하면 계산기 사용
- 최신 정보가 필요하면 검색 사용
- 파일 내용을 알아야 하면 문서 읽기 사용

### 4. Retrievers / RAG
문서 검색 기반 질의응답을 쉽게 구현할 수 있습니다.  
즉, 회사 문서, PDF, 위키, 데이터베이스 등을 검색해서 그 내용을 바탕으로 답변하는 **RAG(Retrieval-Augmented Generation)** 패턴을 지원합니다.

### 5. Memory
대화 내용을 기억